In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated/checkpoints/post_cell_8.pickle

trying: ['file_name_2018', 'file_name_2017']
me:  8
me:  8
trying: ['file_name_2018', 'file_name_2017']
me:  8
me:  8
trying: ['convert_df_of_counts_to_percentages']
me:  4
trying: ['create_dataframe_of_counts']
me:  4
trying: ['go']
me:  0
trying: ['warnings']
me:  0
trying: ['count_then_return_percent']
me:  4
trying: ['title_for_chart']
me:  16
trying: ['sns']
me:  0
trying: ['add_year_column_to_dataframes']
me:  4
trying: ['glob']
me:  0
trying: ['grab_subset_of_data']
me:  4
trying: ['combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions']
me:  6
trying: ['load_survey_data']
me:  6
trying: ['responses_df_2018']


me:  10
trying: ['replace_hyphen_with_en_dash']
me:  10
trying: ['base_dir_2021']
me:  8
trying: ['responses_df_2019']


me:  10
trying: ['file_name_2021']
me:  8
trying: ['base_dir_2019']
me:  8
trying: ['question_of_interest']
me:  16
trying: ['title_for_y_axis']
me:  16
trying: ['base_dir_2018']
me:  8
trying: ['responses_df_2017']


me:  8
trying: ['base_dir_2020']
me:  8
trying: ['base_dir_2017']
me:  8
trying: ['responses_df_2020']


me:  8
trying: ['responses_df_2021']


me:  8
trying: ['directory']
me:  8
trying: ['px']
me:  0
trying: ['combine_subset_of_data_from_multiple_years']
me:  6
trying: ['base_dir_2022']
me:  8
trying: ['responses_df_2022']


me:  16
trying: ['file_name_2022']
me:  8
trying: ['file_name_2019']
me:  8
trying: ['pd']
me:  0
trying: ['file_name_2020']
me:  8
trying: ['orientation_for_chart']
me:  16
trying: ['bar_chart_multiple_choice_17']
me:  16
trying: ['percentages']
me:  16
trying: ['responses_in_order']
me:  16
trying: ['count_then_return_percent_17']
me:  16
trying: ['create_choropleth_map_16']
me:  14
trying: ['np']
me:  0
trying: ['orig_output']
me:  15
trying: ['percentages_per_country_df']
me:  14
trying: ['create_dataframe_of_counts_16']
me:  14
trying: ['bar_chart_multiple_years']
me:  2
trying: ['label_for_legend']
me:  14
trying: ['bar_chart_multiple_choice']
me:  2
trying: ['bar_chart_multiple_choice_multiple_selection']
me:  2
trying: ['responses_per_country_df']
me:  14
trying: ['create_choropleth_map']
me:  2
Declaring variable go
Declaring variable warnings
Declaring variable sns
Declaring variable glob
Declaring variable px
Declaring variable pd
Declaring variable np
Declaring variable bar

In [4]:
%%RecordEvent
%%time
### cell 9 ###

# Optimized cell 9 ###

# 1. Standardize country names in 2022
responses_df_2022.rename(
    columns={
        'United Kingdom (UK)': 'United Kingdom of Great Britain and Northern Ireland'
    }, inplace=True
)
responses_df_2022['In which country do you currently reside?'] = \
    responses_df_2022['In which country do you currently reside?']\
    .replace({'United Kingdom (UK)': 'United Kingdom of Great Britain and Northern Ireland'})

# 2. Standardize country names and job‐title column in 2017
responses_df_2017.rename(
    columns={
        'Country': 'In which country do you currently reside?',
        'CurrentJobTitleSelect': 'Select the title most similar to your current role (or most recent title if retired): - Selected Choice'
    }, inplace=True
)
responses_df_2017['In which country do you currently reside?'] = \
    responses_df_2017['In which country do you currently reside?']\
    .replace({
        'United States': 'United States of America',
        "People 's Republic of China": 'China',
        'United Kingdom': 'United Kingdom of Great Britain and Northern Ireland'
    })

# 3. Consolidate all other countries into 'Other' using a single vectorized pass
for df in [
    responses_df_2017,
    responses_df_2018,
    responses_df_2019,
    responses_df_2020,
    responses_df_2021,
    responses_df_2022
]:
    df['In which country do you currently reside?'] = \
        df['In which country do you currently reside?']\
        .where(df['In which country do you currently reside?'].isin(subset_of_countries), 'Other')

# 4. Optimized combine function, building each year’s DataFrame with reset_index

def combine_subset_of_data_from_multiple_years_18(question_of_interest, x_axis_title, include_2017=False):
    years = [2018, 2019, 2020, 2021, 2022]
    if include_2017:
        years.insert(0, 2017)
    frames = []
    for y in years:
        df_year = globals()[f"responses_df_{y}"]
        s = count_then_return_percent_18(df_year, question_of_interest).sort_index()
        s.name = 'percentage'
        temp = s.reset_index()
        temp.columns = [question_of_interest, 'percentage']
        temp['year'] = str(y)
        frames.append(temp)
    combined = pd.concat(frames, ignore_index=True)
    # rename x-axis column to x_axis_title (may be empty string)
    col = x_axis_title or question_of_interest
    combined = combined.rename(columns={question_of_interest: col})
    return combined[['percentage', 'year', col]]

# 5. Prepare parameters and generate the chart
question_of_interest = 'In which country do you currently reside?'
column_of_interest = 'percentage'
title_for_chart = 'Most Common Nationalities on Kaggle (2017-2022)'
title_for_x_axis = ''
title_for_y_axis = '% of respondents'

country_df_combined = combine_subset_of_data_from_multiple_years_18(
    question_of_interest, title_for_x_axis, include_2017=True
)
country_df_combined = country_df_combined.sort_values(by=['year', 'percentage'], ascending=True)
# unify back to short UK label for presentation
t_country = 'United Kingdom of Great Britain and Northern Ireland'
country_df_combined = country_df_combined.replace(
    {t_country: 'United Kingdom'}
)

bar_chart_multiple_years_18(
    country_df_combined,
    title_for_x_axis,
    column_of_interest,
    title_for_chart,
    title_for_y_axis
)
print("'Other' = any country not shown")

NameError: name 'subset_of_countries' is not defined

In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high/checkpoints/post_cell_9_try_9.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_9_try_9.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[9], f)


In [ ]:
opt_output = Out.get(4)